# UABR — Phase 4: teach `<ANSWER>` vs `<CALL>` (delegation) on top of `base_v1`

**What happens:**
1. **Knowledge labels.** Ask `base_v1` every non-test fact as a cloze ("Kael Dorn was born in the city of"). Correct → ANSWER, otherwise CALL.
2. **Delegation training, two decision formats** (attempt 3, see `docs/results.md`). In both, the decision is made right where the answer word would start, after the system writes the cloze sentence opening:
   - `recite`: `<USER> Where was Kael Dorn born?<ASSISTANT> Kael Dorn was born in the city of` → `<ANSWER>` / `<CALL>`
   - `cloze`: `Kael Dorn was born in the city of` → `<ANSWER>` / `<CALL>`
3. **Gate on dev** for each (six checks), then the **preregistered choice**: `recite` if it passes; else `cloze` if it passes; else the higher dev routing AUROC, reported as not passing.

`QUICK = True` → Run All (~10 min) checks the pipeline. `QUICK = False` → Save & Run All (~15 min).

In [ ]:
QUICK = True
RUN_SECOND_SEED = False   # full run only: repeat training with seed 2027 to check reproducibility (+~15 min)
REPO_URL = "https://github.com/GTX-Pradeep/uabr-slm.git"

import os, glob, json, subprocess
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/uabr_slm"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

data_hits = glob.glob("/kaggle/input/**/data_v1/manifest.json", recursive=True)
base_hits = glob.glob("/kaggle/input/**/base_v1.pt", recursive=True)
assert data_hits, "data_v1 not found: Add Input -> Your Work -> the Phase 2 notebook"
assert base_hits, "base_v1.pt not found: Add Input -> Your Work -> the Phase 3 notebook"
DATA, BASE = os.path.dirname(data_hits[0]), base_hits[0]
KNOW = f"{WORK}/facts_knowledge.parquet"
OUT = f"{WORK}/runs/delegation_quick" if QUICK else f"{WORK}/runs/delegation"
print("DATA :", DATA); print("BASE :", BASE, f"({os.path.getsize(BASE) / 1e6:.0f} MB)"); print("OUT  :", OUT)
m = json.load(open(f"{DATA}/manifest.json")); au = json.load(open(f"{DATA}/audit_report.json"))
print(f"data_v1    : {m['pretrain']['train_tokens']:,} train tokens | reached target: {m['pretrain']['reached_target']}")
print(f"tokenizer  : sha256 {m['tokenizer_sha256'][:12]}")
print(f"audits     : all pass = {au['all_pass']}")
!git log --oneline -1
!python -m pytest tests -q -x

## 1. Knowledge labels from `base_v1` (≈2 min)
Expected pattern, as in the Phase 3 probe: about 75% known at 80 statements, about 16% at 20, about 4% at 5 and when withheld.

In [ ]:
!python scripts/label_knowledge.py --base {BASE} --data {DATA} --out {KNOW}

## 2. Delegation training + dev gate
Prints dev metrics before training and after every epoch, then the full gate.

In [ ]:
quick = "--quick" if QUICK else ""
FORMATS = ["recite", "cloze"]
for f in FORMATS:
    print(f"\n################ format: {f} ################")
    !rm -rf {OUT}_{f}
    !python scripts/train_delegation.py --base {BASE} --data {DATA} --knowledge {KNOW} --out {OUT}_{f} --format {f} {quick}

## 2b. Choose the format (rule fixed before any attempt-3 result)
`recite` if its gate passes; else `cloze` if its gate passes; else the format with the higher dev routing AUROC, reported as **not passing**. The chosen `delegation_v1.pt` is copied to the notebook's output root for Phase 5.

In [ ]:
import shutil
reps = {f: json.load(open(f"{OUT}_{f}/gate_report.json")) for f in FORMATS}
print(f"{'format':8s} {'AUROC':>7s} {'MI bits':>8s} {'acc@10%':>8s} {'CALL rate 0x/80x':>17s}  failed checks")
for f, r in reps.items():
    s = r["summary"]; t = s["call_rate_by_tier"]
    print(f"{f:8s} {r['auroc']:7.3f} {s['decisiveness']['mutual_information']:8.3f} {s['budget_10']['accuracy']:8.3f} "
          f"{t.get('0', t.get(0, float('nan'))):8.2f}/{t.get('80', t.get(80, float('nan'))):.2f}  "
          f"{[k for k, v in r['checks'].items() if not v] or 'none'}")
if reps["recite"]["all_pass"]:
    CHOSEN, why = "recite", "recite passes the gate"
elif reps["cloze"]["all_pass"]:
    CHOSEN, why = "cloze", "recite fails, cloze passes the gate"
else:
    CHOSEN = max(FORMATS, key=lambda f: reps[f]["auroc"]); why = "neither passes: higher dev AUROC (report as NOT passing)"
OUT = f"{OUT}_{CHOSEN}"
shutil.copy(f"{OUT}/delegation_v1.pt", f"{WORK}/delegation_v1.pt")
json.dump({"chosen": CHOSEN, "reason": why, "gate_passed": reps[CHOSEN]["all_pass"],
           "per_format": {f: {"auroc": r["auroc"], "all_pass": r["all_pass"], "checks": r["checks"]} for f, r in reps.items()}},
          open(f"{WORK}/delegation_choice.json", "w"), indent=1, default=float)
print(f"\nCHOSEN: {CHOSEN} ({why}) -> {WORK}/delegation_v1.pt")

## 3. Look at real predictions

In [ ]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 40)
df = pd.read_parquet(f"{OUT}/dev_predictions.parquet")
cols = ["question", "gold", "predicted_action", "call_score", "answer_text", "reward_answer", "call_answer_text", "reward_call"]
print("Known-looking facts the model chose to ANSWER:")
print(df[df.predicted_action == "ANSWER"].sort_values("call_score").head(8)[cols].to_string(index=False))
print("\nFacts the model chose to CALL for:")
print(df[df.predicted_action == "CALL"].sort_values("call_score", ascending=False).head(8)[cols].to_string(index=False))
print("\nBy exposure tier (share routed to CALL, direct accuracy, CALL accuracy):")
print(df.groupby("exposure_count").agg(call_rate=("predicted_action", lambda s: (s == "CALL").mean()),
      direct_acc=("reward_answer", "mean"), call_acc=("reward_call", "mean"), n=("prompt_id", "size")).round(3).to_string())

## 4. Optional: second seed (reproducibility gate)

In [ ]:
if RUN_SECOND_SEED and not QUICK:
    !python scripts/train_delegation.py --base {BASE} --data {DATA} --knowledge {KNOW} --out {WORK}/runs/delegation_seed2027 --format {CHOSEN} --seed 2027
    a = json.load(open(f"{OUT}/gate_report.json")); b = json.load(open(f"{WORK}/runs/delegation_seed2027/gate_report.json"))
    print(f"AUROC seed 2026 {a['auroc']:.3f} vs seed 2027 {b['auroc']:.3f}")
else:
    print("skipped")